# 00a. pandas I: Series, DataFrames, selection

Primer for the course, written by Claude. You don't write this one; you work through it.
Four notebooks, about two hours each:

| | Topic | Needed by |
|---|---|---|
| **00a** | Series, DataFrames, selecting, changing values | week 1 step 2 |
| 00b | Alignment, `axis`, vectorising, iteration, missing data | week 1 step 3 |
| 00c | Time series: shifts, cumulative, rolling, resample, causality | week 2 |
| 00d | Reshape, groupby, merge, linear algebra, the course toolkit | weeks 3-6 |

**How to work through this.** Before running a cell, say out loud what you expect it to
print, including the *shape* and the *labels*. Most pandas confusion is a wrong guess about
which labels come back, and you only notice if you guessed first. Exercises have an empty
cell beneath them and the expected answer above; match it, then move on.

This notebook runs pandas 3.x. Where pandas 3 changed behaviour you may see in older tutorials, it says so.

## Setup

Synthetic prices from `portfolio_lab.data.synthetic`: the same eight tickers as week 1,
business days from 2015-01-01, no network and no `data/` needed, identical on every run
(`seed=7`). The generator returns the store's **long** format; `pivot` turns it into the
**wide** frame the course uses everywhere. Primer 00d explains `pivot`; for now take it on
trust.

The generator's drift is deliberately strong (so the momentum tests have something to
find), so price **levels** wander far from anything real: some tickers end at a fraction of
a cent on the dollar. Use these frames for mechanics, and don't read finance into the numbers.

In [1]:
import numpy as np
import pandas as pd

from portfolio_lab.data.returns import simple_returns
from portfolio_lab.data.synthetic import make_synthetic_prices

pd.set_option("display.precision", 4)
pd.set_option("display.width", 100)

TICKERS = ["SPY", "EFA", "IEF", "GLD", "AAPL", "MSFT", "JNJ", "XOM"]
long, truth = make_synthetic_prices(TICKERS, n_days=2520, seed=7)  # long format, like the store
long["date"] = pd.to_datetime(long["date"])
px = long.pivot(index="date", columns="ticker", values="close")    # wide: dates x tickers
px.columns.name = None
rets = simple_returns(px)                                           # your week-1 function
px.shape, rets.shape

((2520, 8), (2520, 8))

## 1. The one mental model

**A DataFrame is a dict of columns. Each column is a Series. All columns share one index.**

Not a grid, not a list of rows. Three consequences you will meet constantly:

- Grabbing a column is free; building a row means gathering one value from every column.
- Each column has **one dtype**. A row that mixes columns must be coerced to a common one.
- The index is not a row number. It is a set of **labels**, and pandas matches on labels
  whenever two objects meet (primer 00b). If you know SQL: the index is like a primary
  key, except that it takes part in arithmetic.

In [2]:
print(type(px))
print(px.shape)          # (rows, columns)
print(px.index[:3])      # the row labels: dates
print(px.columns)        # the column labels: tickers, sorted alphabetically by pivot
px.dtypes

<class 'pandas.DataFrame'>
(2520, 8)
DatetimeIndex(['2015-01-01', '2015-01-02', '2015-01-05'], dtype='datetime64[s]', name='date', freq=None)
Index(['AAPL', 'EFA', 'GLD', 'IEF', 'JNJ', 'MSFT', 'SPY', 'XOM'], dtype='str')


AAPL    float64
EFA     float64
GLD     float64
IEF     float64
JNJ     float64
MSFT    float64
SPY     float64
XOM     float64
dtype: object

## 2. Series: values with labels

A Series is a 1-D array plus an index of labels, plus a `name`. Build one by hand:

In [3]:
s = pd.Series({"SPY": 0.010, "IEF": -0.002, "GLD": 0.004}, name="2015-01-05")
s

SPY    0.010
IEF   -0.002
GLD    0.004
Name: 2015-01-05, dtype: float64

In [4]:
print(s.index)           # labels
print(s.to_numpy())      # the raw numpy array; labels gone (prefer this to .values)
print(s.name)
print(s["IEF"], s.iloc[0])   # by label, by position

Index(['SPY', 'IEF', 'GLD'], dtype='str')
[ 0.01  -0.002  0.004]
2015-01-05
-0.002 0.01


Arithmetic and comparisons act on every element at once and **keep the labels**. A
comparison gives a boolean Series, which can then select from the original (a *mask*):

In [5]:
print(s * 100)
print(s > 0)
s[s > 0]

SPY    1.0
IEF   -0.2
GLD    0.4
Name: 2015-01-05, dtype: float64
SPY     True
IEF    False
GLD     True
Name: 2015-01-05, dtype: bool


SPY    0.010
GLD    0.004
Name: 2015-01-05, dtype: float64

Reductions collapse a Series to one number. `idxmax` / `idxmin` return the **label** where
the max or min sits, which is usually the thing you actually wanted:

In [6]:
s.max(), s.idxmax(), s.mean(), s.count()

(np.float64(0.01), 'SPY', np.float64(0.004), np.int64(3))

A column of the price frame is exactly this, with dates for labels:

In [7]:
spy = px["SPY"]
print(type(spy), spy.name)
spy.describe()

<class 'pandas.Series'> SPY


count    2520.0000
mean       42.0350
std        42.8440
min         4.5150
25%        10.5381
50%        23.2313
75%        83.5430
max       143.5483
Name: SPY, dtype: float64

**dtypes you will see in this course**

| dtype | holds | example |
|---|---|---|
| `float64` | numbers, and `NaN` for missing | prices, returns |
| `int64` | whole numbers, **cannot hold NaN** | counts |
| `bool` | True/False | masks |
| `datetime64[ns]` (pandas 3 may show `[us]` or `[s]`, the stored resolution) | timestamps | the index |
| `str` | text (new default in pandas 3; older pandas showed `object`) | tickers |

The long frame shows it: `ticker` is `str`, and `date` is `datetime64` only because the
setup cell ran `pd.to_datetime`. Before that it was `object`: plain Python `date`s that
pandas cannot do date arithmetic on. When dates behave oddly, check the dtype first.

In [8]:
long.dtypes

date      datetime64[s]
ticker              str
close           float64
volume          float64
dtype: object

**Exercise 1.** Using `spy`: the date and value of SPY's highest close, and the number of
days SPY closed above 120. Put the three answers in a tuple.

Expected:

```
2015-10-06 00:00:00

143.5483

213
```

In [9]:
spy = px["SPY"]
results = (spy.idxmax(), spy.max(), (spy > 120).sum())
results

(Timestamp('2015-10-06 00:00:00'),
 np.float64(143.54830047629866),
 np.int64(213))

## 3. Looking at a frame

Your first five moves on any new frame. Run them every time; they are how you catch a
wrong pivot or a missing ticker before it becomes a wrong number.

In [10]:
px.head(3)

,AAPL,EFA,GLD,IEF,JNJ,MSFT,SPY,XOM
date,,,,,,,,
2015-01-01,99.5859,100.5228,99.3121,100.6853,100.4359,99.6051,100.7528,100.0591
2015-01-02,98.0027,99.7485,95.5645,99.8149,98.4500,99.5954,97.9358,99.8681
2015-01-05,97.3302,99.3895,95.6451,99.5625,98.0712,98.9247,99.0545,100.4863


In [11]:
px.tail(3)

,AAPL,EFA,GLD,IEF,JNJ,MSFT,SPY,XOM
date,,,,,,,,
2024-08-26,157.0104,608.3428,0.7378,35.7804,10.7119,1646.8116,6.3972,31.4236
2024-08-27,156.8396,608.1834,0.7250,35.4115,10.7150,1623.7468,6.3553,31.5405
2024-08-28,157.5634,617.0917,0.7191,35.7749,10.9412,1621.4877,6.3019,31.0743


In [12]:
px.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2520 entries, 2015-01-01 to 2024-08-28
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2520 non-null   float64
 1   EFA     2520 non-null   float64
 2   GLD     2520 non-null   float64
 3   IEF     2520 non-null   float64
 4   JNJ     2520 non-null   float64
 5   MSFT    2520 non-null   float64
 6   SPY     2520 non-null   float64
 7   XOM     2520 non-null   float64
dtypes: float64(8)
memory usage: 177.2 KB


In [13]:
px.describe().T      # .T transposes: one row per ticker is easier to read

,count,mean,std,min,25%,50%,75%,max
AAPL,2520.0,1367.8820,1265.4077,91.2970,230.6513,826.8169,2461.4675,4079.3392
EFA,2520.0,314.4711,180.6570,66.1834,193.3717,285.8456,417.6200,832.6522
GLD,2520.0,20.6641,31.7013,0.4202,0.8544,2.5233,32.4190,100.4789
IEF,2520.0,103.8013,83.5997,14.8904,37.8125,78.8635,142.8080,333.7636
JNJ,2520.0,65.2269,40.6517,8.2017,21.5843,64.2095,99.5967,156.7431
MSFT,2520.0,1858.3870,2143.8270,48.6294,165.7629,537.4243,3490.9970,7412.0097
SPY,2520.0,42.0350,42.8440,4.5150,10.5381,23.2313,83.5430,143.5483
XOM,2520.0,181.1263,122.9740,31.0743,104.3953,151.2572,208.9362,576.4852


## 4. Selecting

| You want | Write | You get |
|---|---|---|
| one column | `px["SPY"]` | Series |
| several columns | `px[["SPY", "IEF"]]` (a *list* inside) | DataFrame |
| one row by label | `px.loc["2016-03-01"]` | Series indexed by ticker |
| one row by position | `px.iloc[0]` | Series |
| one cell | `px.loc["2016-03-01", "SPY"]`, `px.iloc[0, 0]` | scalar |
| rows by label range | `px.loc["2016-03-01":"2016-03-04"]` | DataFrame, **end included** |
| rows by position range | `px.iloc[0:3]` | DataFrame, end excluded |
| rows by condition | `px[px["SPY"] > 120]` or `px.loc[mask, cols]` | DataFrame |

Rules:

1. **`px[x]` selects columns.** The exceptions are a boolean mask or a slice, which select
   rows. This is the rule behind `prices[i]` raising `KeyError` in your first
   `log_returns`.
2. **`.loc` is labels, `.iloc` is positions.** Both are `[rows, columns]`.
3. **`.loc` slices include the end label**; `.iloc` slices exclude it, as Python does.
4. With a `DatetimeIndex`, a partial date string selects the whole period:
   `px.loc["2016-03"]` is every day in March 2016, `px.loc["2016"]` the whole year.

In [14]:
px.loc["2016-03-01"]              # one row -> Series indexed by ticker, name = the date

AAPL    130.8472
EFA      69.1082
GLD      76.3689
IEF     223.8553
JNJ      88.7966
MSFT     64.8242
SPY     120.7976
XOM     204.3601
Name: 2016-03-01 00:00:00, dtype: float64

In [15]:
px.loc["2016-03-01":"2016-03-04", ["SPY", "IEF"]]   # both ends included

,SPY,IEF
date,,
2016-03-01,120.7976,223.8553
2016-03-02,121.6984,223.2333
2016-03-03,121.6550,222.5213
2016-03-04,122.0018,222.9555


In [16]:
px.iloc[0:3, 0:2]                 # positions: rows 0,1,2 and columns 0,1

,AAPL,EFA
date,,
2015-01-01,99.5859,100.5228
2015-01-02,98.0027,99.7485
2015-01-05,97.3302,99.3895


In [17]:
px.loc["2016-03", "SPY"].head()   # partial string: all of March 2016

date
2016-03-01    120.7976
2016-03-02    121.6984
2016-03-03    121.6550
2016-03-04    122.0018
2016-03-07    123.3849
Name: SPY, dtype: float64

**Masks.** Combine conditions with `&` (and), `|` (or), `~` (not), and **wrap each
condition in parentheses**, because `&` binds tighter than `>`. Python's `and`/`or` raise
an error on a Series: they want a single True/False, and a Series holds many.

In [18]:
both_up = (rets["SPY"] > 0) & (rets["IEF"] > 0)
print(both_up.sum(), "days both rose")      # True counts as 1
rets.loc[both_up, ["SPY", "IEF"]].head(3)

751 days both rose


,SPY,IEF
date,,
2015-01-08,0.0126,0.0186
2015-01-15,0.0089,0.0244
2015-01-16,0.0088,0.0166


In [19]:
rets.loc[rets["SPY"].between(-0.001, 0.001), "SPY"].count()   # flat-ish days

np.int64(134)

`.at` and `.iat` are the fast single-cell versions of `.loc` and `.iloc`. Use them inside
the rare loop that really needs one cell at a time.

In [20]:
px.at[pd.Timestamp("2016-03-01"), "SPY"], px.iat[0, 0]

(np.float64(120.79756782766637), np.float64(99.58594654387602))

**Exercise 2.** The AAPL and MSFT closes for March 2016. Answer with the number of rows
and the first row (as a Series).

Expected:

```
23

AAPL    130.8472
MSFT     64.8242
Name: 2016-03-01 00:00:00, dtype: float64
```

In [21]:
mar = px.loc["2016-03", ["AAPL", "MSFT"]]
len(mar), mar.iloc[0]

(23,
 AAPL    130.8472
 MSFT     64.8242
 Name: 2016-03-01 00:00:00, dtype: float64)

**Exercise 3.** On how many days did **both** SPY and GLD close above their own
full-sample mean close? (Each ticker is compared with its own mean.)

Expected:

```
699
```

In [22]:
both = (px["SPY"] > px["SPY"].mean()) & (px["GLD"] > px["GLD"].mean())
both.sum()

np.int64(699)

## 5. Changing values (pandas 3: Copy-on-Write)

Since pandas 3, **every selection behaves like a copy**. You change a frame by one indexing
call *on that frame*, never through an intermediate selection:

In [23]:
demo = px[["SPY", "IEF"]].head(3)

demo["SPY"].iloc[0] = 0.0                # chained: edits a temporary Series, not demo
print(demo.iloc[0, 0], "<- unchanged")

demo.loc[demo.index[0], "SPY"] = 0.0     # one .loc call on demo itself: works
print(demo.iloc[0, 0], "<- changed")

100.75279783260551 <- unchanged
0.0 <- changed


/var/folders/n1/bpj6pv2x27q2t32y9937j7pm0000gp/T/ipykernel_78192/2716306995.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  demo["SPY"].iloc[0] = 0.0                # chained: edits a temporary Series, not demo


pandas 3 warns about the chained version (`ChainedAssignmentError`) but doesn't stop it,
so the line runs and silently changes nothing. Read that warning as an error.

Older tutorials show `SettingWithCopyWarning` and sprinkle `.copy()` everywhere. Both are
gone. The rule is simpler now: **one `.loc`/`.iloc` on the object you want changed.**
`px` is also untouched by the cell above: `demo` was a copy the moment it was selected.

Adding and removing columns:

In [24]:
demo["spread"] = demo["SPY"] - demo["IEF"]           # new column, in place
demo = demo.assign(ratio=demo["SPY"] / demo["IEF"])  # returns a NEW frame; reassign it
demo = demo.drop(columns=["ratio"]).rename(columns={"spread": "spy_minus_ief"})
demo

,SPY,IEF,spy_minus_ief
date,,,
2015-01-01,0.0000,100.6853,-100.6853
2015-01-02,97.9358,99.8149,-1.8791
2015-01-05,99.0545,99.5625,-0.5080


Almost every pandas method **returns a new object** and leaves the original alone:
`drop`, `rename`, `sort_values`, `fillna`, `assign`. If nothing seemed to happen, you
probably forgot to assign the result. Skip `inplace=True`; it saves nothing and breaks
method chaining.

**Exercise 4.** Build `small` from the first five rows of SPY and IEF. Add a column
`spread = SPY - IEF`. Then set IEF on the third row to `NaN` with one `.loc` call, **after**
computing `spread` (so `spread` keeps its old value). Show `small`.

Expected:

```
                 SPY       IEF  spread
date                                  
2015-01-01  100.7528  100.6853  0.0675
2015-01-02   97.9358   99.8149 -1.8791
2015-01-05   99.0545       NaN -0.5080
2015-01-06   96.9184   98.2850 -1.3666
2015-01-07   94.7434   99.0726 -4.3292
```

In [25]:
small = px[["SPY", "IEF"]].head(5)
small = small.assign(spread=small["SPY"] - small["IEF"])
small.loc[small.index[2], "IEF"] = np.nan
small

,SPY,IEF,spread
date,,,
2015-01-01,100.7528,100.6853,0.0675
2015-01-02,97.9358,99.8149,-1.8791
2015-01-05,99.0545,NaN,-0.5080
2015-01-06,96.9184,98.2850,-1.3666
2015-01-07,94.7434,99.0726,-4.3292


## What to remember

- Frame = dict of Series sharing an index. Labels, not positions, are the identity.
- `px["x"]` is a column; `.loc` is labels; `.iloc` is positions; `.loc` slices include the
  end.
- Masks: `&`, `|`, `~`, parentheses around each condition.
- Change a frame with one `.loc` on it. Methods return new objects; assign the result.
- Always look at `shape`, `head`, `dtypes` before trusting a frame.

## Reading for this notebook

- McKinney, *Python for Data Analysis*, 3rd ed., ch. 5 "Getting Started with pandas".
  Free online at https://wesmckinney.com/book/. Written by pandas' creator; the standard
  text in university data-science syllabi.
- VanderPlas, *Python Data Science Handbook*, ch. 3 sections "Introducing Pandas Objects"
  and "Data Indexing and Selection": https://jakevdp.github.io/PythonDataScienceHandbook/
- pandas user guide, "Indexing and selecting data" and "Copy-on-Write (CoW)":
  https://pandas.pydata.org/docs/user_guide/index.html
- pandas 3.0 release notes, for what changed from older tutorials:
  https://pandas.pydata.org/docs/whatsnew/v3.0.0.html